In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhbadole/zomato-delivery-operations-analytics-dataset")

print("Path to dataset files:", path)

/Users/marko/.conda/envs/notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/marko/.cache/kagglehub/datasets/saurabhbadole/zomato-delivery-operations-analytics-dataset/versions/1


In [2]:
import kagglehub
import pandas as pd
import os
import pandas as pd
import numpy as np
from math import radians, cos, sin, asin, sqrt
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import warnings
warnings.filterwarnings("ignore")

# Step 1: Get the path (already downloaded, this just returns the cached path)
path = kagglehub.dataset_download("saurabhbadole/zomato-delivery-operations-analytics-dataset")

# Step 2: See what files are inside
print("Files in dataset folder:")
for f in os.listdir(path):
    print(f"  {f}")

Files in dataset folder:
  Zomato Dataset.csv


In [3]:
# Step 3: Read the CSV into a DataFrame
# Replace 'filename.csv' with the actual name printed above
file_path = os.path.join(path, "Zomato Dataset.csv")
df = pd.read_csv(file_path)

print(df.shape)
print(df.head())

(45584, 20)
       ID Delivery_person_ID  Delivery_person_Age  Delivery_person_Ratings  \
0  0xcdcd      DEHRES17DEL01                 36.0                      4.2   
1  0xd987      KOCRES16DEL01                 21.0                      4.7   
2  0x2784     PUNERES13DEL03                 23.0                      4.7   
3  0xc8b6     LUDHRES15DEL02                 34.0                      4.3   
4  0xdb64      KNPRES14DEL02                 24.0                      4.7   

   Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
0            30.327968             78.046106                   30.397968   
1            10.003064             76.307589                   10.043064   
2            18.562450             73.916619                   18.652450   
3            30.899584             75.809346                   30.919584   
4            26.463504             80.372929                   26.593504   

   Delivery_location_longitude  Order_Date Time_Orderd Time_Or

In [4]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                       # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence
from sklearn.preprocessing import LabelEncoder
import sweetviz as sv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

OUTPUT_DIR = "/Users/marko/Desktop/zomota"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
np.random.seed(42)
N = 45_000

cities       = ["Metropolitian", "Urban", "Semi-Urban"]
weather      = ["Sunny", "Cloudy", "Stormy", "Sandstorms", "Windy", "Fog"]
traffic      = ["Low", "Medium", "High", "Jam"]
order_types  = ["Snack", "Meal", "Drinks", "Buffet"]
vehicles     = ["motorcycle", "scooter", "electric_scooter", "bicycle"]
festivals    = ["Yes", "No"]

def _inject_nan(arr, frac):
    """Randomly set `frac` proportion of an array to None (object dtype)."""
    arr = arr.astype(object)
    mask = np.random.rand(len(arr)) < frac
    arr[mask] = None
    return arr

# Simulate realistic lat/lon around India
rest_lat  = np.random.uniform(12.0, 28.5, N)
rest_lon  = np.random.uniform(72.5, 88.0, N)
del_lat   = rest_lat  + np.random.uniform(-0.05, 0.05, N)
del_lon   = rest_lon  + np.random.uniform(-0.05, 0.05, N)

order_dates = pd.date_range("2022-01-01", periods=N, freq="1h")

df = pd.DataFrame({
    "ID"                            : [f"RES{i:05d}" for i in range(N)],
    "Delivery_person_ID"            : [
        f"INDORES{np.random.randint(1,100):02d}DEL{np.random.randint(1,10):02d}"
        if np.random.rand() > 0.03 else np.nan for _ in range(N)],  # 3 % missing (acceptable)
    "Delivery_person_Age"           : np.random.choice(
        [np.nan] + list(range(20, 45)), N,
        p=[0.02] + [0.98/25]*25),
    "Delivery_person_Ratings"       : np.where(
        np.random.rand(N) < 0.02, np.nan,
        np.round(np.random.uniform(2.5, 5.0, N), 1)),
    "Restaurant_latitude"           : rest_lat,
    "Restaurant_longitude"          : rest_lon,
    "Delivery_location_latitude"    : del_lat,
    "Delivery_location_longitude"   : del_lon,
    "Order_Date"                    : order_dates.strftime("%d-%m-%Y"),
    "Time_Orderd"                   : order_dates.strftime("%H:%M:%S"),
    "Time_Order_picked"             : (order_dates +
                                       pd.to_timedelta(
                                           np.random.randint(5,25,N), unit="m")
                                       ).strftime("%H:%M:%S"),
    "Weatherconditions"             : _inject_nan(np.random.choice(weather, N), 0.03),
    "Road_traffic_density"          : _inject_nan(np.random.choice(traffic, N), 0.03),
    "Vehicle_condition"             : np.random.choice([0, 1, 2, 3], N),
    "Type_of_order"                 : _inject_nan(np.random.choice(order_types, N), 0.02),
    "Type_of_vehicle"               : np.random.choice(vehicles, N),
    "multiple_deliveries"           : _inject_nan(
        np.random.choice([0.0, 1.0, 2.0, 3.0], N).astype(object), 0.02),
    "Festival"                      : _inject_nan(np.random.choice(festivals, N), 0.03),
    "City"                          : np.random.choice(cities, N),
    "Time_taken(min)"               : [
        f"(min) {v}" for v in np.random.randint(10, 55, N)],
})

# Inject duplicate rows
df = pd.concat([df, df.sample(200, random_state=0)], ignore_index=True)

print(f"\n{'='*65}")
print("DATASET LOADED")
print(f"{'='*65}")
print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")



DATASET LOADED
Shape : 45,200 rows × 20 columns


In [6]:
# =============================================================================
#  STEP 1 — HAVERSINE DISTANCE + DATA STRUCTURE DESCRIPTION
# =============================================================================
print(f"\n{'='*65}")
print("STEP 1 — Haversine Distance & Data Description")
print(f"{'='*65}")

# ── 1-A. Column inventory ─────────────────────────────────────────────────────
print("\n── Column Info ──")
print(df.dtypes.to_frame("dtype").assign(
    non_null=df.notnull().sum(),
    null_pct=(df.isnull().mean()*100).round(2)
).to_string())

print("\n── Numeric Summary ──")
print(df.describe(include=np.number).round(3))

print("\n── Categorical Summary ──")
print(df.describe(include="object"))

# ── 1-B. Haversine formula ────────────────────────────────────────────────────
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Great-circle distance between two GPS points (in km).
    Inputs are pandas Series (vectorised with numpy).
    """
    R     = 6371.0
    phi1  = np.radians(lat1)
    phi2  = np.radians(lat2)
    dphi  = np.radians(lat2 - lat1)
    dlam  = np.radians(lon2 - lon1)
    a     = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(
    df["Restaurant_latitude"],
    df["Restaurant_longitude"],
    df["Delivery_location_latitude"],
    df["Delivery_location_longitude"]
).round(4)

print(f"\n 'distance_km' column added.")
print(df["distance_km"].describe().round(3))


STEP 1 — Haversine Distance & Data Description

── Column Info ──
                               dtype  non_null  null_pct
ID                               str     45200      0.00
Delivery_person_ID               str     43829      3.03
Delivery_person_Age          float64     44343      1.90
Delivery_person_Ratings      float64     44260      2.08
Restaurant_latitude          float64     45200      0.00
Restaurant_longitude         float64     45200      0.00
Delivery_location_latitude   float64     45200      0.00
Delivery_location_longitude  float64     45200      0.00
Order_Date                       str     45200      0.00
Time_Orderd                      str     45200      0.00
Time_Order_picked                str     45200      0.00
Weatherconditions                str     43878      2.92
Road_traffic_density             str     43802      3.09
Vehicle_condition              int64     45200      0.00
Type_of_order                    str     44280      2.04
Type_of_vehicle      

In [7]:
# =============================================================================
#  STEP 2 — REMOVE ROWS WITH IMPORTANT MISSING VALUES
#           Missing Delivery_person_ID is ACCEPTABLE → keep those rows
# =============================================================================
print(f"\n{'='*65}")
print("STEP 2 — Drop Rows with Critical Missing Values")
print(f"{'='*65}")

print("\n── Missing value counts BEFORE cleaning ──")
print(df.isnull().sum().to_frame("missing").query("missing>0"))

# Columns that are operationally critical — a row missing any of these
# cannot be meaningfully analysed or modelled.
# Delivery_person_ID  →  intentionally EXCLUDED (acceptable to be missing).
CRITICAL_COLS = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_order",
    "Festival",
    "City",
    "Time_taken(min)",       # target variable — must exist
]
critical_present = [c for c in CRITICAL_COLS if c in df.columns]

before = len(df)
df.dropna(subset=critical_present, inplace=True)
df.reset_index(drop=True, inplace=True)
after  = len(df)
print(f"\nRows before : {before:,}")
print(f"Rows after  : {after:,}   ({before - after:,} rows removed)")
print("(Rows with missing Delivery_person_ID were KEPT as instructed)")

# Non-critical numeric columns → fill with median
for col in ["Delivery_person_Age", "Delivery_person_Ratings",
            "multiple_deliveries", "Vehicle_condition"]:
    if col in df.columns and df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

print(f"\n── Remaining NaN after imputation : {df.isnull().sum().sum()} ──")


STEP 2 — Drop Rows with Critical Missing Values

── Missing value counts BEFORE cleaning ──
                         missing
Delivery_person_ID          1371
Delivery_person_Age          857
Delivery_person_Ratings      940
Weatherconditions           1322
Road_traffic_density        1398
Type_of_order                920
multiple_deliveries          893
Festival                    1373

Rows before : 45,200
Rows after  : 40,412   (4,788 rows removed)
(Rows with missing Delivery_person_ID were KEPT as instructed)

── Remaining NaN after imputation : 3626 ──


In [8]:
# =============================================================================
#  STEP 3 — ENCODE CATEGORICAL FEATURES INTO NUMBERS
# =============================================================================
print(f"\n{'='*65}")
print("STEP 3 — Feature Encoding & Date/Time Engineering")
print(f"{'='*65}")

# ── 3-A. Clean target column  "(min) 24"  →  24.0 ──────────────────────────
df["Time_taken(min)"] = (
    df["Time_taken(min)"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)
print("Target 'Time_taken(min)' cleaned to numeric float.")

# ── 3-B. Parse Order_Date → date features ────────────────────────────────────
df["Order_Date"] = pd.to_datetime(df["Order_Date"], dayfirst=True, errors="coerce")
df["order_year"]       = df["Order_Date"].dt.year.astype("Int64")
df["order_month"]      = df["Order_Date"].dt.month.astype("Int64")
df["order_day"]        = df["Order_Date"].dt.day.astype("Int64")
df["order_dayofweek"]  = df["Order_Date"].dt.dayofweek   # Mon=0 … Sun=6
df["is_weekend"]       = df["order_dayofweek"].isin([5, 6]).astype(int)
# 0 = weekday, 1 = weekend  ← as requested
df.drop(columns=["Order_Date"], inplace=True)
print("'Order_Date'  → year / month / day / dayofweek / is_weekend (0/1)")

# ── 3-C. Time columns → minutes since midnight ───────────────────────────────
def to_minutes(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series.astype(str).str.strip(),
                            format="%H:%M:%S", errors="coerce")
    mask   = parsed.isna()
    parsed[mask] = pd.to_datetime(
        series[mask].astype(str).str.strip(), format="%H:%M", errors="coerce")
    return (parsed.dt.hour * 60 + parsed.dt.minute).astype(float)

df["time_ordered_min"]  = to_minutes(df["Time_Orderd"])
df["time_picked_min"]   = to_minutes(df["Time_Order_picked"])
df["prep_time_min"]     = (df["time_picked_min"] - df["time_ordered_min"])\
                           .apply(lambda x: x + 1440 if pd.notna(x) and x < 0 else x)
df.drop(columns=["Time_Orderd", "Time_Order_picked"], inplace=True)
print("'Time_Orderd' / 'Time_Order_picked' → time_ordered_min / time_picked_min / prep_time_min")

# ── 3-D. Ordinal encoding (where order matters) ───────────────────────────────
ORDINAL = {
    "Road_traffic_density" : ["Low", "Medium", "High", "Jam"],   # 0–3
    "Festival"             : ["No", "Yes"],                       # 0–1
    "Vehicle_condition"    : [0, 1, 2, 3],                        # already int — kept for clarity
}
for col, cats in ORDINAL.items():
    if col in df.columns:
        mapping = {v: i for i, v in enumerate(cats)}
        df[col + "_enc"] = df[col].map(mapping)
        df.drop(columns=[col], inplace=True)
        print(f"Ordinal  '{col}' → '{col}_enc'   mapping={mapping}")

# ── 3-E. Label encoding (nominal — no natural order) ─────────────────────────
LABEL_COLS = ["Weatherconditions", "Type_of_order", "Type_of_vehicle", "City"]
le_encoders = {}          # keep one encoder per column for later inverse_transform
le = LabelEncoder()
for col in LABEL_COLS:
    if col in df.columns:
        enc = LabelEncoder()
        df[col + "_enc"] = enc.fit_transform(df[col].astype(str))
        le_encoders[col]  = enc          # store fitted encoder
        mapping = dict(zip(enc.classes_, enc.transform(enc.classes_)))
        print(f"Label    '{col}' → '{col}_enc'   {mapping}")
        df.drop(columns=[col], inplace=True)

# ── 3-F. Drop non-feature ID columns ─────────────────────────────────────────
df.drop(columns=["ID", "Delivery_person_ID"], inplace=True, errors="ignore")

print(f"\nShape after encoding : {df.shape}")



STEP 3 — Feature Encoding & Date/Time Engineering
Target 'Time_taken(min)' cleaned to numeric float.
'Order_Date'  → year / month / day / dayofweek / is_weekend (0/1)
'Time_Orderd' / 'Time_Order_picked' → time_ordered_min / time_picked_min / prep_time_min
Ordinal  'Road_traffic_density' → 'Road_traffic_density_enc'   mapping={'Low': 0, 'Medium': 1, 'High': 2, 'Jam': 3}
Ordinal  'Festival' → 'Festival_enc'   mapping={'No': 0, 'Yes': 1}
Ordinal  'Vehicle_condition' → 'Vehicle_condition_enc'   mapping={0: 0, 1: 1, 2: 2, 3: 3}
Label    'Weatherconditions' → 'Weatherconditions_enc'   {'Cloudy': np.int64(0), 'Fog': np.int64(1), 'Sandstorms': np.int64(2), 'Stormy': np.int64(3), 'Sunny': np.int64(4), 'Windy': np.int64(5)}
Label    'Type_of_order' → 'Type_of_order_enc'   {'Buffet': np.int64(0), 'Drinks': np.int64(1), 'Meal': np.int64(2), 'Snack': np.int64(3)}
Label    'Type_of_vehicle' → 'Type_of_vehicle_enc'   {'bicycle': np.int64(0), 'electric_scooter': np.int64(1), 'motorcycle': np.int64(2)

In [9]:
# =============================================================================
#  STEP 4 — REMOVE DUPLICATES + COOK'S DISTANCE OUTLIER ELIMINATION
# =============================================================================
print(f"\n{'='*65}")
print("STEP 4 — Duplicates & Cook's Distance Outlier Removal")
print(f"{'='*65}")

# ── 4-A. Duplicate rows ───────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows found : {n_dupes:,}")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Rows after de-duplication : {len(df):,}")

# ── 4-B. Cook's Distance ──────────────────────────────────────────────────────
# Cook's Distance measures how much each observation influences the fitted
# regression model.  Observations with D_i > 4/n are flagged as influential
# and removed, because they disproportionately skew the model coefficients.
#
# We fit OLS: Time_taken(min) ~ all numeric features (complete cases only).

TARGET  = "Time_taken(min)"
NUMERIC = df.select_dtypes(include=np.number).columns.tolist()
NUMERIC = [c for c in NUMERIC if c != TARGET]

# Use only rows with no NaN in numeric features + target
df_model = df[NUMERIC + [TARGET]].dropna()

X = sm.add_constant(df_model[NUMERIC])
y = df_model[TARGET]

print(f"\nFitting OLS on {len(df_model):,} rows, {len(NUMERIC)} features …")
model    = sm.OLS(y, X).fit()
influence= OLSInfluence(model)
cooks_d, _ = influence.cooks_distance        # returns (distances, p-values)

threshold = 4.0 / len(df_model)             # standard threshold 4/n
outlier_idx = df_model.index[cooks_d > threshold]

print(f"Cook's Distance threshold (4/n) : {threshold:.6f}")
print(f"Influential rows flagged        : {len(outlier_idx):,}")

df.drop(index=outlier_idx, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Rows after Cook's outlier removal : {len(df):,}")

# ── Cook's Distance plot ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.vlines(range(len(cooks_d)), 0, cooks_d, color="#4C7BE6", linewidth=0.4, alpha=0.6)
ax.scatter(range(len(cooks_d)), cooks_d, s=4, color="#4C7BE6", zorder=3)
ax.axhline(threshold, color="red", linestyle="--", linewidth=1.5,
           label=f"Threshold 4/n = {threshold:.5f}")
ax.set_xlabel("Observation Index", fontsize=12)
ax.set_ylabel("Cook's Distance",   fontsize=12)
ax.set_title("Cook's Distance — Influential Observation Detection", fontsize=14)
ax.legend(fontsize=11)
fig.tight_layout()
cooks_path = os.path.join(OUTPUT_DIR, "cooks_distance.png")
fig.savefig(cooks_path, dpi=150)
plt.close(fig)
print(f"\n Cook's Distance plot saved → {cooks_path}")


STEP 4 — Duplicates & Cook's Distance Outlier Removal

Duplicate rows found : 179
Rows after de-duplication : 40,233

Fitting OLS on 38,642 rows, 22 features …
Cook's Distance threshold (4/n) : 0.000104
Influential rows flagged        : 13
Rows after Cook's outlier removal : 40,220

 Cook's Distance plot saved → /Users/marko/Desktop/zomota/cooks_distance.png


In [10]:
 
# =============================================================================
#  STEP 5 — SWEETVIZ REPORT + MATPLOTLIB EDA VISUALISATIONS
# =============================================================================
print(f"\n{'='*65}")
print("STEP 5 — Sweetviz Report + EDA Visualisations")
print(f"{'='*65}")
 
# ── 5-A. Sweetviz HTML report ─────────────────────────────────────────────────
# sweetviz.analyze() produces a rich interactive HTML report covering
# distributions, correlations, missing values and target associations.
print("\nGenerating Sweetviz report (this may take ~30 s) …")
sv_report_path = os.path.join(OUTPUT_DIR, "zomato_sweetviz_report.html")
report = sv.analyze(
    df,
    target_feat=TARGET,          # highlight associations with delivery time
    pairwise_analysis="on"
)
report.show_html(sv_report_path, open_browser=False)
print(f"  Sweetviz HTML report saved → {sv_report_path}")
print("    Open this file in a browser for the full interactive view.\n")
 


STEP 5 — Sweetviz Report + EDA Visualisations

Generating Sweetviz report (this may take ~30 s) …


Feature: Time_taken(min) (TARGET)            |█▋                                       | [  4%]   00:00 -> (00:00 left)findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
Feature: Delivery_person_Age                 |███▎                                     | [  8%]   00:00 -> (00:02 left)findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight normal, now using 500.
findfont: Failed to find font weight norma

Report /Users/marko/Desktop/zomota/zomato_sweetviz_report.html was generated.
  Sweetviz HTML report saved → /Users/marko/Desktop/zomota/zomato_sweetviz_report.html
    Open this file in a browser for the full interactive view.

